# Component 2: PV Fault Detection, Colab environment check (Phase 6)

**Member 2: PV-Specific On-Device TinyML Fault Detection**, branch `component2-tinyml`

This notebook checks that Google Colab works with the repository code **before any model training**. The cells below are the ones that were executed step by step in Colab. The result is recorded in `results/colab_env_check.json`.

It does:
- get the repository code;
- check the uploaded dataset;
- load the saved run-wise splits;
- build the 8 PV features for **one** cross-validation fold.

It does **not**:
- train a model (Phase 7);
- use OPERATION-window data for fitting (it only counts its rows);
- re-implement any formula or split. Everything comes from `component2-tinyml/src/tinyml/`.

**How to run:** click a cell, press **Shift + Enter**, and go in order from Step 1. In Step 4 a **Choose Files** button appears: upload `pv_fault_dataset.csv`. A CPU runtime is enough.

## Step 2 (run this first): get the repository and install dependencies
Clones the `component2-tinyml` branch if it is not there yet (for example after a runtime reset), then installs `requirements.txt` and prints the package versions.

In [ ]:
# STEP 2: (Re)check repository, install dependencies, show versions
import os, sys, subprocess

REPO_URL = "https://github.com/Lisura123/J26-IT-348"
BRANCH = "component2-tinyml"
REPO_DIR = "/content/J26-IT-348"
COMPONENT_DIR = f"{REPO_DIR}/component2-tinyml"

if not os.path.isdir(REPO_DIR):
    print("Runtime was reset: cloning the repository again")
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR], check=True)

commit = subprocess.run(["git", "-C", REPO_DIR, "log", "-1", "--format=%h %s"],
                        capture_output=True, text=True, check=True).stdout.strip()
print("Commit:", commit)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                f"{COMPONENT_DIR}/requirements.txt"], check=True)

import numpy as np, pandas as pd, sklearn, pytest
print("Python      :", sys.version.split()[0])
print("numpy       :", np.__version__)
print("pandas      :", pd.__version__)
print("scikit-learn:", sklearn.__version__)
print("pytest      :", pytest.__version__)
print("Install OK")

## Step 1b: Verify branch and commit
Expected: `Branch OK : True`. `Commit OK` checks the commit this check was first run on (`3c1f1a4`); a later commit on the branch prints `False` there, which is expected.

In [ ]:
# STEP 1b: Verify branch and commit
import os, subprocess

REPO_DIR = "/content/J26-IT-348"
EXPECTED_BRANCH = "component2-tinyml"
EXPECTED_COMMIT = "3c1f1a4"

def git(*args):
    return subprocess.run(["git", "-C", REPO_DIR, *args],
                          capture_output=True, text=True, check=True).stdout.strip()

branch = git("branch", "--show-current")
commit = git("log", "-1", "--format=%h %s")
print("Branch :", branch)
print("Commit :", commit)
print("Branch OK :", branch == EXPECTED_BRANCH)
print("Commit OK :", commit.startswith(EXPECTED_COMMIT))
print("Component folder exists:", os.path.isdir(os.path.join(REPO_DIR, "component2-tinyml")))

## Step 3: Import the repository modules
Formulas, splits and checks are imported from the repository and are not copied into this notebook.

In [ ]:
# STEP 3: Import the repository modules (no formulas are copied into this notebook)
import sys
SRC_DIR = "/content/J26-IT-348/component2-tinyml/src"
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

from tinyml.contract import LABELS
from tinyml.validate import validate_dataset, file_sha256
from tinyml.splits import load_assignments, iter_folds
from tinyml.features import FEATURES, FEATURE_UNITS, FeatureConfig, fold_matrices, input_status
import tinyml.features

print("Modules loaded from:", tinyml.features.__file__)
print("Fault classes      :", LABELS)
print("Feature version    :", FeatureConfig().version)
print("Model input order  :")
for i, (f, u) in enumerate(zip(FEATURES, FEATURE_UNITS)):
    print(f"  {i}. {f} [{u}]")
print("Import OK")

## Step 4: Upload the dataset
Click **Choose Files** and select `pv_fault_dataset.csv` (expected size 957,366 bytes). The file stays in this Colab session only and is never uploaded to GitHub.

In [ ]:
# STEP 4: Upload pv_fault_dataset.csv from your computer
import os
from google.colab import files

uploaded = files.upload()
csv_files = [name for name in uploaded if name.endswith(".csv")]
if not csv_files:
    raise SystemExit("No CSV uploaded. Run this cell again and choose pv_fault_dataset.csv")

DATA_PATH = os.path.abspath(csv_files[0])
print("Dataset file:", DATA_PATH)
print("Size (bytes):", os.path.getsize(DATA_PATH))

## Step 5: Check the dataset fingerprint (SHA-256)
The dataset must be byte-identical to the one the Phase 4 splits were made from. Otherwise the notebook stops.

In [ ]:
# STEP 5: Verify the dataset SHA-256 against the Phase 4 split configuration
import json

SPLITS_PATH = "/content/J26-IT-348/component2-tinyml/results/splits_runwise_v1.json"
with open(SPLITS_PATH, encoding="utf-8") as f:
    split_meta = json.load(f)["meta"]

expected_sha = split_meta["dataset_sha256"]
actual_sha = file_sha256(DATA_PATH)

print("Split version   :", split_meta["split_version"])
print("Expected SHA-256:", expected_sha)
print("Your file       :", actual_sha)
if actual_sha != expected_sha:
    raise SystemExit("STOP: this dataset does not match the one used for the Phase 4 splits.")
print("SHA-256 OK: dataset matches the split configuration.")

## Step 6: Dataset shape, class distribution and integrity
Read-only. Expected: shape (6000, 20), 600 readings and 10 runs per class and window, **0 errors and 4 warnings**. The `health_score` leakage accuracy differed between runs: 0.6945 in Colab (scikit-learn 1.6.1) and 0.7457 locally (scikit-learn 1.9.1). The version difference is the likely cause but has not been confirmed. Both values are far above chance (≈ 0.2).

In [ ]:
# STEP 6: Dataset shape, class distribution and data integrity (read-only)
import pandas as pd

df = pd.read_csv(DATA_PATH)
print("Dataset shape (rows, columns):", df.shape)
print()

rows = pd.crosstab(df["label"], df["window"]).reindex(list(LABELS))
runs = pd.crosstab(df.drop_duplicates("run_id")["label"],
                   df.drop_duplicates("run_id")["window"]).reindex(list(LABELS))
print("Readings per class and window:")
print(rows.to_string())
print()
print("Runs per class and window:")
print(runs.to_string())
print()

report = validate_dataset(df)
print("Integrity check -> errors:", len(report["errors"]), "| warnings:", len(report["warnings"]))
for e in report["errors"]:
    print("  ERROR:", e)
for w in report["warnings"]:
    print("  -", w)

## Step 7: Load the saved Phase 4 run-wise splits
No new splitting. Checks that no run is in both train and test, and that CV uses TRAINING runs only.

In [ ]:
# STEP 7: Load the saved Phase 4 run-wise splits (no new splitting)
assignments, meta = load_assignments(SPLITS_PATH, df, dataset_sha256=actual_sha)
print("Split version:", meta["split_version"], "| seed:", meta["seed"],
      "| runs:", meta["runs"], "| made with scikit-learn", meta["sklearn_version"])
print()

split_check = {}
for protocol in ("cv", "temporal", "loso"):
    n_folds = 0
    for name, tr, te in iter_folds(df, assignments, protocol):
        shared = set(df.loc[tr, "run_id"]) & set(df.loc[te, "run_id"])
        assert not shared, f"{name}: runs in both train and test: {shared}"
        n_folds += 1
    split_check[protocol] = n_folds
    print(f"{protocol:9s}: {n_folds} folds, no run shared between train and test")

cv_runs = assignments[assignments["cv_fold"].astype(int) >= 0]
print()
print("CV uses only TRAINING runs:", set(cv_runs["window"]) == {"TRAINING"})
print("CV runs per fold and class:")
print(pd.crosstab(cv_runs["cv_fold"], cv_runs["label"]).reindex(columns=list(LABELS)).to_string())

## Step 8: Feature smoke test on one CV fold
TRAINING window only. `V_ref` and the scaler are fitted on the fold's training runs and applied, without refitting, to its validation runs. No model is trained.

In [ ]:
# STEP 8: Feature smoke test on one CV fold (TRAINING window only, no model training)
import numpy as np

train_window = df[df["window"] == "TRAINING"]      # OPERATION is not used here
FOLD = "cv0"
fold_name, train_idx, val_idx = next((n, tr, te) for n, tr, te
                                     in iter_folds(train_window, assignments, "cv") if n == FOLD)

m = fold_matrices(train_window, train_idx, val_idx, fold=fold_name)   # fit on train runs only
cfg = m["config"]
Xtr, Xval = m["train"]["X"], m["test"]["X"]

print("Fold                  :", fold_name)
print("Training runs / rows  :", train_window.loc[train_idx, "run_id"].nunique(), "/", len(train_idx))
print("Validation runs / rows:", train_window.loc[val_idx, "run_id"].nunique(), "/", len(val_idx))
print("Training matrix shape :", Xtr.shape, Xtr.dtype)
print("Validation matrix shape:", Xval.shape, Xval.dtype)
print("Feature names         :", list(cfg.features))
print("V_ref (fitted on training runs):", {k: round(v, 4) for k, v in cfg.vref.items()})
print("Fitted on             :", cfg.fitted_on)
print()

for side, title in (("train", "Training part"), ("test", "Validation part")):
    status = m[side]["status"]
    table = pd.crosstab(train_window.loc[status.index, "label"], status).reindex(list(LABELS)).fillna(0).astype(int)
    print(f"Quality gates, {title} (readings per class):")
    print(table.to_string())
    print()

checks = {
    "8 features in agreed order": list(cfg.features) == list(FEATURES),
    "training inputs finite": bool(np.isfinite(Xtr).all()),
    "validation inputs finite": bool(np.isfinite(Xval).all()),
    "inputs are float32": Xtr.dtype == np.float32 and Xval.dtype == np.float32,
    "fitted on training rows only": cfg.fitted_on["train_rows"] == len(train_idx),
    "no run in both train and validation": not (set(train_window.loc[train_idx, "run_id"])
                                                & set(train_window.loc[val_idx, "run_id"])),
    "OPERATION rows not used": not (df.loc[list(train_idx) + list(val_idx), "window"] == "OPERATION").any(),
    "training features scaled (mean~0, std~1)": bool(np.allclose(Xtr.mean(0), 0, atol=1e-3)
                                                     and np.allclose(Xtr.std(0), 1, atol=1e-2)),
}
for name, ok in checks.items():
    print(("PASS  " if ok else "FAIL  ") + name)

## Step 9: Run the repository tests and save a summary
Runs the repository tests in Colab and downloads `colab_env_check.json`.

In [ ]:
# STEP 9: Run the repository tests and save a summary of this Colab check
import json, subprocess, sys

result = subprocess.run([sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider"],
                        cwd=COMPONENT_DIR, capture_output=True, text=True)
tests_line = result.stdout.strip().splitlines()[-1] if result.stdout.strip() else result.stderr[-300:]
print(result.stdout[-1500:])
print("Test result:", tests_line)

summary = {
    "commit": commit,
    "python": sys.version.split()[0],
    "numpy": np.__version__, "pandas": pd.__version__, "sklearn": sklearn.__version__,
    "dataset_sha256_ok": actual_sha == expected_sha,
    "dataset_shape": list(df.shape),
    "validation_errors": len(report["errors"]),
    "validation_warnings": len(report["warnings"]),
    "health_score_runwise_accuracy": report["leakage"]["single_column_runwise_accuracy"],
    "split_folds": split_check,
    "fold": fold_name,
    "train_matrix_shape": list(Xtr.shape),
    "validation_matrix_shape": list(Xval.shape),
    "vref": cfg.vref,
    "feature_checks": {k: bool(v) for k, v in checks.items()},
    "tests": tests_line,
}
with open("/content/colab_env_check.json", "w") as f:
    json.dump(summary, f, indent=1)
print(json.dumps(summary, indent=1))

from google.colab import files
files.download("/content/colab_env_check.json")